# ANTARES Cumulative Nightly History

This notebook builds the RSP/platform-backed LSST-only nightly history store without replacing the normal `alerts_time_comparison.ipynb` workflow.


In [1]:
from pathlib import Path
import sys

REPO_URL = 'https://github.com/darim1151/ANTARES_Analysis.git'
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    %cd /content
    !rm -rf ANTARES_Analysis
    !git clone {REPO_URL}
    %cd ANTARES_Analysis
    !git pull --ff-only
    PROJECT_ROOT = Path('/content/ANTARES_Analysis')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('src files:', sorted(path.name for path in (PROJECT_ROOT / 'src').iterdir()))


Project root: /home/mdarim/notebooks/ANTARES_Analysis
src files: ['__init__.py', '__pycache__', 'cache.py', 'chunked_query.py', 'config.py', 'figures.py', 'history.py', 'lightcurves.py', 'query.py', 'summary.py', 'validation.py']


In [2]:
%pip install --quiet antares-client elasticsearch-dsl astropy matplotlib pandas numpy pyarrow


Note: you may need to restart the kernel to use updated packages.


In [3]:
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Not running in Colab; using platform/local storage from src/config.py.')


Not running in Colab; using platform/local storage from src/config.py.


In [4]:
from src import config, history

DATA_ROOT = config.HISTORY_DATA_ROOT
MJD_HISTORY_START = config.MJD2_MIN
MJD_HISTORY_CUTOFF = config.MJD2_MAX

print('Ready.')
config.print_config_summary()
print()
print('History store:', DATA_ROOT)
print('Survey data root:', history.survey_data_root(DATA_ROOT))


Ready.
Configuration
  Last Night: MJD 61181.0 - 61182.0  (1.0 days)  [OK]
  Cumulative LSST History: MJD 61095.0 - 61181.0  (86.0 days)  [OK]
  Samples per range : 5000
  Survey mode       : lsst
  LSST-only filter  : ON
  LSST history start: MJD 61095.0
  Tag filter        : none (all alerts)
  Random seed       : 42
  Realtime night    : ON
  ANTARES lookback  : 1 day(s)
  Populated search  : ON
  Search depth      : 5 day(s)
  Chunked ingest    : ON
  Chunk start size  : 1 day(s)
  Chunk min size    : 30 sec
  Chunk split at    : 9,500/10,000 loci
  Parallel shards   : 3
  History backfill  : OFF
  History data root : /project/mdarim/ANTARES_Analysis
  History data set  : lsst_only
  History target    : 100,000 loci/night
  History LC fetch  : ON
  Use stored history: ON

  Ranges are NON-overlapping  (MJD2_MAX=61181.0, MJD1_MIN=61181.0)

History store: /project/mdarim/ANTARES_Analysis
Survey data root: /project/mdarim/ANTARES_Analysis/data/lsst_only


## 1. Tiny Smoke Backfill

This creates one nightly partition with a small target and skips lightcurves. Use it first to verify platform paths, manifests, parquet writing, and resume behavior.


In [5]:
from pathlib import Path

DATA_ROOT = Path("/home/mdarim/ANTARES_Analysis_Data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print("Using writable DATA_ROOT:", DATA_ROOT)

Using writable DATA_ROOT: /home/mdarim/ANTARES_Analysis_Data


In [7]:
smoke_manifest = history.ingest_night(
    data_root=DATA_ROOT,
    mjd_min=61096.0,
    mjd_max=61096.1,
    target_loci=100,
    fetch_lightcurves=False,
    resume=False,
    range_label="History 2026/2/25 smoke",
    parallel_shards=1,
    max_results_per_chunk=100,
    split_threshold=95,
    lsst_only=True,
    update_indexes=True,
    verbose=True,
)

smoke_manifest



  Chunked query 'History 2026/2/25 smoke'  MJD [61096.000000, 61096.100000]
    ES limit=100, split at >= 95, minimum chunk=30s
       1. 61096.000000-61096.100000    8640.0s     100 loci  split  (live; 2 queued)
       2. 61096.000000-61096.050000    4320.0s     100 loci  split  (live; 3 queued)
       3. 61096.000000-61096.025000    2160.0s     100 loci  split  (live; 4 queued)
       4. 61096.000000-61096.012500    1080.0s       0 loci  accepted  (live; 3 queued)
       5. 61096.012500-61096.025000    1080.0s     100 loci  split  (live; 4 queued)
       6. 61096.012500-61096.018750     540.0s       0 loci  accepted  (live; 3 queued)
       7. 61096.018750-61096.025000     540.0s     100 loci  split  (live; 4 queued)
       8. 61096.018750-61096.021875     270.0s       0 loci  accepted  (live; 3 queued)
       9. 61096.021875-61096.025000     270.0s     100 loci  split  (live; 4 queued)
      10. 61096.021875-61096.023438     135.0s       0 loci  accepted  (live; 3 queued)
      11. 

{'manifest': {'date_utc': '2026-02-25',
  'mjd_min': 61096.0,
  'mjd_max': 61096.1,
  'query_tag': None,
  'target_loci': 100,
  'actual_loci': 100,
  'alert_rows': 0,
  'chunk_count': 6,
  'split_count': 8,
  'saturated_chunk_count': 0,
  'status': 'complete',
  'survey_mode': 'lsst',
  'lsst_filter_used': True,
  'lsst_filter': {'bool': {'should': [{'exists': {'field': 'properties.survey.lsst.dia_object_id'}},
     {'exists': {'field': 'properties.survey.lsst.ss_object_id'}}],
    'minimum_should_match': 1}},
  'parallel_shards': 1,
  'lsst_dia_count': 100,
  'lsst_ss_count': 0,
  'ztf_object_id_count': 0,
  'started_at_utc': '2026-05-23T22:04:45+00:00',
  'finished_at_utc': '2026-05-23T22:04:53+00:00',
  'runtime_seconds': 7.9,
  'validation': {'mjd_pass': True,
   'mjd_missing_column': False,
   'mjd_below_count': 0,
   'mjd_above_count': 0,
   'duplicate_locus_count': 0,
   'coordinate_pass': True,
   'coordinate_missing_columns': False,
   'bad_ra_count': 0,
   'bad_dec_count': 0

In [8]:
loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

print("Cumulative loci rows:", len(loci_index))
display(nightly_summary.tail())


Cumulative loci rows: 100


,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,lsst_ss_count,ztf_object_id_count,lsst_only_pass,history_start_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
0,2026-02-24,2026/2/24,61095.0,61096.0,None,1000,0,0,0,0,...,0,0,None,None,112.07,2026-05-11T23:05:28+00:00,2026-05-11T23:07:20+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
1,2026-02-25,2026/2/25,61096.0,61096.1,None,100,100,0,6,8,...,0,0,True,True,7.90,2026-05-23T22:04:45+00:00,2026-05-23T22:04:53+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...


In [9]:
test_manifest = history.ingest_night(
    data_root=DATA_ROOT,
    mjd_min=61096.0,
    mjd_max=61096.25,
    target_loci=1000,
    fetch_lightcurves=False,
    resume=False,
    range_label="History 2026/2/25 quarter-night test",
    parallel_shards=1,
    max_results_per_chunk=500,
    split_threshold=475,
    lsst_only=True,
    update_indexes=True,
    verbose=True,
)

test_manifest["manifest"]


  Chunked query 'History 2026/2/25 quarter-night test'  MJD [61096.000000, 61096.250000]
    ES limit=500, split at >= 475, minimum chunk=30s
       1. 61096.000000-61096.250000   21600.0s     500 loci  split  (live; 2 queued)
       2. 61096.000000-61096.125000   10800.0s     500 loci  split  (live; 3 queued)
       3. 61096.000000-61096.062500    5400.0s     500 loci  split  (live; 4 queued)
       4. 61096.000000-61096.031250    2700.0s     500 loci  split  (live; 5 queued)
       5. 61096.000000-61096.015625    1350.0s       0 loci  accepted  (live; 4 queued)
       6. 61096.015625-61096.031250    1350.0s     500 loci  split  (live; 5 queued)
       7. 61096.015625-61096.023438     675.0s       0 loci  accepted  (live; 4 queued)
       8. 61096.023438-61096.031250     675.0s     500 loci  split  (live; 5 queued)
       9. 61096.023438-61096.027344     337.5s     500 loci  split  (live; 6 queued)
      10. 61096.023438-61096.025391     168.8s     311 loci  accepted  (live; 5 queued)

{'date_utc': '2026-02-25',
 'mjd_min': 61096.0,
 'mjd_max': 61096.25,
 'query_tag': None,
 'target_loci': 1000,
 'actual_loci': 1000,
 'alert_rows': 0,
 'chunk_count': 6,
 'split_count': 8,
 'saturated_chunk_count': 0,
 'status': 'complete',
 'survey_mode': 'lsst',
 'lsst_filter_used': True,
 'lsst_filter': {'bool': {'should': [{'exists': {'field': 'properties.survey.lsst.dia_object_id'}},
    {'exists': {'field': 'properties.survey.lsst.ss_object_id'}}],
   'minimum_should_match': 1}},
 'parallel_shards': 1,
 'lsst_dia_count': 1000,
 'lsst_ss_count': 0,
 'ztf_object_id_count': 0,
 'started_at_utc': '2026-05-23T22:05:01+00:00',
 'finished_at_utc': '2026-05-23T22:05:42+00:00',
 'runtime_seconds': 41.5,
 'validation': {'mjd_pass': True,
  'mjd_missing_column': False,
  'mjd_below_count': 0,
  'mjd_above_count': 0,
  'duplicate_locus_count': 0,
  'coordinate_pass': True,
  'coordinate_missing_columns': False,
  'bad_ra_count': 0,
  'bad_dec_count': 0,
  'overlap_count': 0,
  'alert_locus_

In [ ]:
loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

print("Cumulative loci rows:", len(loci_index))
display(nightly_summary.tail())


In [ ]:
half_manifest = history.ingest_night(
    data_root=DATA_ROOT,
    mjd_min=61096.0,
    mjd_max=61096.5,
    target_loci=5000,
    fetch_lightcurves=False,
    resume=False,
    range_label="History 2026/2/25 half-night test",
    parallel_shards=1,
    max_results_per_chunk=1000,
    split_threshold=950,
    lsst_only=True,
    update_indexes=True,
    verbose=True,
)

half_manifest["manifest"]


In [ ]:
from pathlib import Path
from src import history

DATA_ROOT = Path("/home/mdarim/ANTARES_Analysis_Data_Clean")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

night_ranges = [
    (61095.0, 61096.0, "2026/2/24"),
    (61096.0, 61097.0, "2026/2/25"),
    (61097.0, 61098.0, "2026/2/26"),
    (61098.0, 61099.0, "2026/2/27"),
    (61099.0, 61100.0, "2026/2/28"),
]

for lo, hi, label in night_ranges:
    print("\n" + "=" * 72)
    print(f"Running LSST-only ANTARES backfill for {label}")

    result = history.ingest_night(
        data_root=DATA_ROOT,
        mjd_min=lo,
        mjd_max=hi,
        target_loci=1000,
        fetch_lightcurves=False,
        resume=True,
        range_label=f"History {label}",
        parallel_shards=1,
        max_results_per_chunk=500,
        split_threshold=475,
        lsst_only=True,
        update_indexes=True,
        verbose=True,
    )

    print(result["manifest"]["status"], result["manifest"]["actual_loci"])

loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

print("\nDONE")
print("DATA_ROOT:", DATA_ROOT)
print("Cumulative LSST-only loci rows:", len(loci_index))
display(nightly_summary.tail(10))



In [ ]:
resume_summary = history.backfill_history(
    data_root=DATA_ROOT,
    mjd_start=MJD_HISTORY_START,
    mjd_stop=MJD_HISTORY_START + 1,
    target_loci=1000,
    max_nights=1,
    fetch_lightcurves=False,
    resume=True,
        parallel_shards=config.CHUNK_PARALLEL_SHARDS,
        lsst_only=config.LSST_ONLY,
)
display(resume_summary.tail())


## 2. Three-Night Test

Turn this on after the smoke test. It checks multi-night folder layout and cumulative-index behavior.


In [ ]:
RUN_THREE_NIGHT_TEST = False

if RUN_THREE_NIGHT_TEST:
    three_night_summary = history.backfill_history(
        data_root=DATA_ROOT,
        mjd_start=MJD_HISTORY_START,
        mjd_stop=MJD_HISTORY_START + 3,
        target_loci=5000,
        max_nights=3,
        fetch_lightcurves=False,
        resume=True,
        parallel_shards=config.CHUNK_PARALLEL_SHARDS,
        lsst_only=config.LSST_ONLY,
    )
    display(three_night_summary.tail(10))


## 3. Full-Scale Runs

Run one full night first. Only then turn on the full historical backfill.


In [12]:
import requests
import time

url = "https://api.antares.noirlab.edu/v1/loci"

for i in range(5):
    print("Test", i + 1)
    try:
        r = requests.get(url, timeout=20)
        print("status:", r.status_code, "seconds ok")
    except Exception as e:
        print("FAILED:", type(e).__name__, e)
    time.sleep(5)

Test 1
status: 200 seconds ok
Test 2
status: 200 seconds ok
Test 3
status: 200 seconds ok
Test 4
status: 200 seconds ok
Test 5
status: 200 seconds ok


In [13]:
from pathlib import Path
from src import history

DATA_ROOT = Path("/home/mdarim/ANTARES_Analysis_Data_Clean")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

test = history.ingest_night(
    data_root=DATA_ROOT,
    mjd_min=61096.0,
    mjd_max=61096.1,
    target_loci=100,
    fetch_lightcurves=False,
    resume=True,
    range_label="History 2026/2/25 smoke",
    parallel_shards=1,
    max_results_per_chunk=100,
    split_threshold=95,
    lsst_only=True,
    update_indexes=True,
    verbose=True,
)

test["manifest"]

  Chunked query 'History 2026/2/25 smoke'  MJD [61096.000000, 61096.100000]
    ES limit=100, split at >= 95, minimum chunk=30s
       1. 61096.000000-61096.100000    8640.0s     100 loci  split  (live; 2 queued)
       2. 61096.000000-61096.050000    4320.0s     100 loci  split  (live; 3 queued)
       3. 61096.000000-61096.025000    2160.0s     100 loci  split  (live; 4 queued)
       4. 61096.000000-61096.012500    1080.0s       0 loci  accepted  (live; 3 queued)
       5. 61096.012500-61096.025000    1080.0s     100 loci  split  (live; 4 queued)
       6. 61096.012500-61096.018750     540.0s       0 loci  accepted  (live; 3 queued)
       7. 61096.018750-61096.025000     540.0s     100 loci  split  (live; 4 queued)
       8. 61096.018750-61096.021875     270.0s       0 loci  accepted  (live; 3 queued)
       9. 61096.021875-61096.025000     270.0s     100 loci  split  (live; 4 queued)
      10. 61096.021875-61096.023438     135.0s       0 loci  accepted  (live; 3 queued)
      11. 

{'date_utc': '2026-02-25',
 'mjd_min': 61096.0,
 'mjd_max': 61096.1,
 'query_tag': None,
 'target_loci': 100,
 'actual_loci': 100,
 'alert_rows': 0,
 'chunk_count': 6,
 'split_count': 8,
 'saturated_chunk_count': 0,
 'status': 'complete',
 'survey_mode': 'lsst',
 'lsst_filter_used': True,
 'lsst_filter': {'bool': {'should': [{'exists': {'field': 'properties.survey.lsst.dia_object_id'}},
    {'exists': {'field': 'properties.survey.lsst.ss_object_id'}}],
   'minimum_should_match': 1}},
 'parallel_shards': 1,
 'lsst_dia_count': 100,
 'lsst_ss_count': 0,
 'ztf_object_id_count': 0,
 'started_at_utc': '2026-05-23T22:20:39+00:00',
 'finished_at_utc': '2026-05-23T22:20:47+00:00',
 'runtime_seconds': 7.34,
 'validation': {'mjd_pass': True,
  'mjd_missing_column': False,
  'mjd_below_count': 0,
  'mjd_above_count': 0,
  'duplicate_locus_count': 0,
  'coordinate_pass': True,
  'coordinate_missing_columns': False,
  'bad_ra_count': 0,
  'bad_dec_count': 0,
  'overlap_count': 0,
  'alert_locus_link

In [11]:
from pathlib import Path
from src import history

DATA_ROOT = Path("/home/mdarim/ANTARES_Analysis_Data_Clean")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

night_ranges = [
    (61095.0, 61096.0, "2026/2/24"),
    (61096.0, 61097.0, "2026/2/25"),
    (61097.0, 61098.0, "2026/2/26"),
    (61098.0, 61099.0, "2026/2/27"),
    (61099.0, 61100.0, "2026/2/28"),
]

for lo, hi, label in night_ranges:
    print("\n" + "=" * 72)
    print(f"Running LSST-only ANTARES backfill for {label}")

    result = history.ingest_night(
        data_root=DATA_ROOT,
        mjd_min=lo,
        mjd_max=hi,
        target_loci=1000,
        fetch_lightcurves=False,
        resume=True,
        range_label=f"History {label}",
        parallel_shards=1,
        max_results_per_chunk=500,
        split_threshold=475,
        lsst_only=True,
        update_indexes=True,
        verbose=True,
    )

    print(result["manifest"]["date_utc"], result["manifest"]["status"], result["manifest"]["actual_loci"])

loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

print("\nDONE")
print("DATA_ROOT:", DATA_ROOT)
print("Cumulative LSST-only loci rows:", len(loci_index))
display(nightly_summary.tail(10))



Running LSST-only ANTARES backfill for 2026/2/24
  Chunked query 'History 2026/2/24'  MJD [61095.000000, 61096.000000]
    ES limit=500, split at >= 475, minimum chunk=30s
       1. 61095.000000-61096.000000   86400.0s     500 loci  split  (live; 2 queued)
       2. 61095.000000-61095.500000   43200.0s     500 loci  split  (live; 3 queued)
       3. 61095.000000-61095.250000   21600.0s     500 loci  split  (live; 4 queued)
       4. 61095.000000-61095.125000   10800.0s     500 loci  split  (live; 5 queued)
       5. 61095.000000-61095.062500    5400.0s     500 loci  split  (live; 6 queued)
       6. 61095.000000-61095.031250    2700.0s     500 loci  split  (live; 7 queued)
       7. 61095.000000-61095.015625    1350.0s       0 loci  accepted  (live; 6 queued)
       8. 61095.015625-61095.031250    1350.0s     500 loci  split  (live; 7 queued)
       9. 61095.015625-61095.023438     675.0s     482 loci  split  (live; 8 queued)
      10. 61095.015625-61095.019531     337.5s       0 loci

ConnectTimeout: HTTPConnectionPool(host='api.antares.noirlab.edu', port=80): Max retries exceeded with url: /v1/loci?sort=-properties.newest_alert_observation_time&elasticsearch_query%5Blocus_listing%5D=%7B%22query%22%3A+%7B%22bool%22%3A+%7B%22filter%22%3A+%5B%7B%22range%22%3A+%7B%22properties.newest_alert_observation_time%22%3A+%7B%22gte%22%3A+61095.021484375%2C+%22lt%22%3A+61095.0234375%7D%7D%7D%2C+%7B%22bool%22%3A+%7B%22should%22%3A+%5B%7B%22exists%22%3A+%7B%22field%22%3A+%22properties.survey.lsst.dia_object_id%22%7D%7D%2C+%7B%22exists%22%3A+%7B%22field%22%3A+%22properties.survey.lsst.ss_object_id%22%7D%7D%5D%2C+%22minimum_should_match%22%3A+1%7D%7D%5D%7D%7D%7D&page%5Boffset%5D=160&page%5Blimit%5D=10 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x7bf970759b20>, 'Connection to api.antares.noirlab.edu timed out. (connect timeout=60)'))

In [14]:
from pathlib import Path
from src import history

DATA_ROOT = Path("/home/mdarim/ANTARES_Analysis_Data_Clean")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

single = history.ingest_night(
    data_root=DATA_ROOT,
    mjd_min=61096.0,
    mjd_max=61097.0,
    target_loci=1000,
    fetch_lightcurves=False,
    resume=True,
    range_label="History 2026/2/25 single-night 1000",
    parallel_shards=1,
    max_results_per_chunk=100,
    split_threshold=95,
    lsst_only=True,
    update_indexes=True,
    verbose=True,
)

single["manifest"]

  Resume: found 2026-02-25; loading existing nightly partition.


{'actual_loci': 100,
 'alert_rows': 0,
 'chunk_count': 6,
 'date_utc': '2026-02-25',
 'finished_at_utc': '2026-05-23T22:20:47+00:00',
 'lsst_dia_count': 100,
 'lsst_filter': {'bool': {'minimum_should_match': 1,
   'should': [{'exists': {'field': 'properties.survey.lsst.dia_object_id'}},
    {'exists': {'field': 'properties.survey.lsst.ss_object_id'}}]}},
 'lsst_filter_used': True,
 'lsst_ss_count': 0,
 'mjd_max': 61096.1,
 'mjd_min': 61096.0,
 'parallel_shards': 1,
 'paths': {'alerts': '/home/mdarim/ANTARES_Analysis_Data_Clean/data/lsst_only/nightly/2026/02/25/alerts.parquet',
  'loci': '/home/mdarim/ANTARES_Analysis_Data_Clean/data/lsst_only/nightly/2026/02/25/loci.parquet',
  'manifest': '/home/mdarim/ANTARES_Analysis_Data_Clean/data/lsst_only/nightly/2026/02/25/manifest.json'},
 'query_tag': None,
 'runtime_seconds': 7.34,
 'saturated_chunk_count': 0,
 'split_count': 8,
 'started_at_utc': '2026-05-23T22:20:39+00:00',
 'status': 'complete',
 'survey_mode': 'lsst',
 'target_loci': 100

In [18]:
from pathlib import Path
import time
import pandas as pd

from src import history


def backfill_range(
    data_root,
    mjd_start,
    mjd_stop,
    target_loci_per_night=1000,
    fetch_lightcurves=False,
    lsst_only=True,
    parallel_shards=1,
    max_retries=5,
    retry_sleep_seconds=(30, 90, 180, 300, 600),
    max_results_per_chunk=100,
    split_threshold=95,
    update_indexes_each_night=False,
    verbose=True,
):
    data_root = Path(data_root)
    data_root.mkdir(parents=True, exist_ok=True)

    chunk_cache_dir = data_root / "cache" / "chunks"
    chunk_cache_dir.mkdir(parents=True, exist_ok=True)

    manifests = []
    failures = []

    for date_utc, lo, hi in history.iter_night_windows(mjd_start, mjd_stop):
        label_display = history.display_date(date_utc)
        range_label = f"History {label_display}"

        print("\n" + "=" * 72)
        print(f"Backfilling {label_display}  MJD [{lo}, {hi}]")

        success = False
        last_error = None

        for attempt in range(1, max_retries + 1):
            print(f"\nAttempt {attempt}/{max_retries} for {label_display}")

            try:
                result = history.ingest_night(
                    data_root=data_root,
                    mjd_min=lo,
                    mjd_max=hi,
                    target_loci=target_loci_per_night,
                    fetch_lightcurves=fetch_lightcurves,
                    resume=True,
                    range_label=range_label,
                    chunk_cache_dir=chunk_cache_dir,
                    parallel_shards=parallel_shards,
                    max_results_per_chunk=max_results_per_chunk,
                    split_threshold=split_threshold,
                    lsst_only=lsst_only,
                    update_indexes=update_indexes_each_night,
                    verbose=verbose,
                )

                manifest = result["manifest"]
                manifests.append(manifest)

                print(
                    f"\nSUCCESS {label_display}: "
                    f"{manifest['status']}  actual_loci={manifest['actual_loci']}"
                )

                success = True
                break

            except Exception as exc:
                last_error = str(exc)
                print(f"\nFAILED attempt {attempt}/{max_retries} for {label_display}")
                print(last_error)

                if attempt < max_retries:
                    sleep_for = retry_sleep_seconds[min(attempt - 1, len(retry_sleep_seconds) - 1)]
                    print(f"Sleeping {sleep_for} seconds before retry...")
                    time.sleep(sleep_for)

        if not success:
            failures.append(
                {
                    "date_utc": date_utc,
                    "display_date": label_display,
                    "mjd_min": lo,
                    "mjd_max": hi,
                    "error": last_error,
                }
            )

    loci_index, nightly_summary = history.update_cumulative_indexes(data_root)

    print("\n" + "=" * 72)
    print("BACKFILL FINISHED")
    print("Successful night attempts:", len(manifests))
    print("Failed nights:", len(failures))
    print("Cumulative loci rows:", len(loci_index))

    return {
        "manifests": manifests,
        "failures": failures,
        "loci_index": loci_index,
        "nightly_summary": nightly_summary,
    }

In [19]:
result = backfill_range(
    data_root=Path("/home/mdarim/ANTARES_Analysis_Data"),
    mjd_start=61095.0,
    mjd_stop=61100.0,
    target_loci_per_night=1000,
    fetch_lightcurves=False,
    lsst_only=True,
    parallel_shards=1,
    max_retries=5,
    max_results_per_chunk=100,
    split_threshold=95,
    update_indexes_each_night=False,
    verbose=True,
)

display(result["nightly_summary"])
result["failures"]


Backfilling 2026/2/24  MJD [61095.0, 61096.0]

Attempt 1/5 for 2026/2/24
  Chunked query 'History 2026/2/24'  MJD [61095.000000, 61096.000000]
    ES limit=100, split at >= 95, minimum chunk=30s
       1. 61095.000000-61096.000000   86400.0s     100 loci  split  (live; 2 queued)
       2. 61095.000000-61095.500000   43200.0s     100 loci  split  (live; 3 queued)
       3. 61095.000000-61095.250000   21600.0s     100 loci  split  (live; 4 queued)
       4. 61095.000000-61095.125000   10800.0s     100 loci  split  (live; 5 queued)
       5. 61095.000000-61095.062500    5400.0s     100 loci  split  (live; 6 queued)
       6. 61095.000000-61095.031250    2700.0s     100 loci  split  (live; 7 queued)
       7. 61095.000000-61095.015625    1350.0s       0 loci  accepted  (live; 6 queued)
       8. 61095.015625-61095.031250    1350.0s     100 loci  split  (live; 7 queued)
       9. 61095.015625-61095.023438     675.0s     100 loci  split  (live; 8 queued)
      10. 61095.015625-61095.019531 

,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,lsst_ss_count,ztf_object_id_count,lsst_only_pass,history_start_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
0,2026-02-24,2026/2/24,61095.0,61096.00,None,1000,1000,0,27,33,...,0,0,True,True,37.20,2026-05-24T09:02:52+00:00,2026-05-24T09:03:29+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
1,2026-02-25,2026/2/25,61096.0,61096.25,None,1000,1000,0,6,8,...,0,0,True,True,41.50,2026-05-23T22:05:01+00:00,2026-05-23T22:05:42+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
2,2026-02-26,2026/2/26,61097.0,61098.00,None,1000,1000,0,20,25,...,0,0,True,True,37.47,2026-05-24T09:03:29+00:00,2026-05-24T09:04:06+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
3,2026-02-27,2026/2/27,61098.0,61099.00,None,1000,1000,0,22,29,...,0,0,True,True,122.44,2026-05-24T09:04:07+00:00,2026-05-24T09:06:09+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
4,2026-02-28,2026/2/28,61099.0,61100.00,None,1000,1000,0,33,39,...,284,3,True,True,49.32,2026-05-24T09:06:09+00:00,2026-05-24T09:06:58+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...


[]

In [22]:
from pathlib import Path
import json
import time
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from antares_client.search import search as antares_search, get_by_id

from src import history, query


def _now_utc():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def _sleep_for(attempt):
    return [20, 60, 120, 240, 480][min(attempt - 1, 4)]


def _query_loci_once(mjd_min, mjd_max, include_upper, ra_min, ra_max, limit, lsst_only=True):
    upper_op = "lte" if include_upper else "lt"

    filters = [
        {
            "range": {
                "properties.newest_alert_observation_time": {
                    "gte": float(mjd_min),
                    upper_op: float(mjd_max),
                }
            }
        },
        {"range": {"ra": {"gte": float(ra_min), "lt": float(ra_max)}}},
    ]

    if lsst_only:
        filters.append(query.lsst_identifier_filter())

    q = {"query": {"bool": {"filter": filters}}}

    rows = []
    for locus in antares_search(q):
        rows.append(query.locus_to_record(locus))
        if len(rows) >= limit:
            break

    return pd.DataFrame(rows)


def _query_loci_retry(*args, max_retries=6, **kwargs):
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            return _query_loci_once(*args, **kwargs)
        except Exception as exc:
            last_error = exc
            print(f"      query failed attempt {attempt}/{max_retries}: {exc}")
            if attempt < max_retries:
                wait = _sleep_for(attempt)
                print(f"      sleeping {wait}s")
                time.sleep(wait)
    raise last_error


def full_loci_for_night(
    data_root,
    date_utc,
    mjd_min,
    mjd_max,
    lsst_only=True,
    max_results_per_chunk=100,
    split_threshold=95,
    min_chunk_seconds=30.0,
    min_ra_degrees=0.05,
    max_retries=6,
):
    cache_dir = Path(data_root) / "cache" / "full_loci_chunks" / date_utc
    cache_dir.mkdir(parents=True, exist_ok=True)

    pending = [(float(mjd_min), float(mjd_max), True, 0.0, 360.0)]
    accepted = []
    report = []
    saturated_leaf_count = 0
    attempt = 0

    while pending:
        lo, hi, include_upper, ra0, ra1 = pending.pop(0)
        attempt += 1
        width_seconds = (hi - lo) * 86400.0
        ra_width = ra1 - ra0

        cache_name = (
            f"mjd_{lo:.8f}_{hi:.8f}_"
            f"upper_{int(include_upper)}_ra_{ra0:.4f}_{ra1:.4f}.parquet"
        ).replace(".", "p")
        cache_path = cache_dir / cache_name

        if cache_path.exists():
            df = pd.read_parquet(cache_path)
            source = "cache"
        else:
            df = _query_loci_retry(
                lo,
                hi,
                include_upper,
                ra0,
                ra1,
                max_results_per_chunk,
                lsst_only=lsst_only,
                max_retries=max_retries,
            )
            source = "live"

        n = len(df)
        saturated = n >= split_threshold

        if saturated and width_seconds > min_chunk_seconds * 1.01:
            mid = (lo + hi) / 2.0
            pending.insert(0, (mid, hi, include_upper, ra0, ra1))
            pending.insert(0, (lo, mid, False, ra0, ra1))
            status = "split_time"

        elif saturated and ra_width > min_ra_degrees * 1.01:
            mid_ra = (ra0 + ra1) / 2.0
            pending.insert(0, (lo, hi, include_upper, mid_ra, ra1))
            pending.insert(0, (lo, hi, include_upper, ra0, mid_ra))
            status = "split_ra"

        else:
            if source == "live":
                df.to_parquet(cache_path, index=False)
            accepted.append(df)
            status = "accepted_saturated" if saturated else "accepted"
            if saturated:
                saturated_leaf_count += 1

        report.append({
            "mjd_min": lo,
            "mjd_max": hi,
            "include_upper": include_upper,
            "ra_min": ra0,
            "ra_max": ra1,
            "width_seconds": width_seconds,
            "n_loci": n,
            "status": status,
            "source": source,
        })

        print(
            f"    {attempt:>5}. MJD {lo:.6f}-{hi:.6f} "
            f"RA {ra0:7.2f}-{ra1:7.2f} "
            f"{n:>5} loci {status} ({source}; {len(pending)} queued)"
        )

    if accepted:
        loci = pd.concat(accepted, ignore_index=True, sort=False)
        loci = loci.drop_duplicates("locus_id", keep="last").reset_index(drop=True)
    else:
        loci = pd.DataFrame()

    return loci, pd.DataFrame(report), saturated_leaf_count


def _fetch_one_lightcurve_cached(locus_id, label, cache_dir, max_retries=6):
    cache_path = Path(cache_dir) / f"{locus_id}.parquet"
    if cache_path.exists():
        return pd.read_parquet(cache_path), None

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            locus = get_by_id(locus_id)
            lc = locus.lightcurve
            if lc is None or lc.empty:
                empty = pd.DataFrame({"locus_id": [locus_id], "range_label": [label], "empty_lightcurve": [True]})
                empty.to_parquet(cache_path, index=False)
                return empty, None

            lc = lc.copy()
            lc["locus_id"] = locus_id
            lc["range_label"] = label
            lc.to_parquet(cache_path, index=False)
            return lc, None

        except Exception as exc:
            last_error = str(exc)
            if attempt < max_retries:
                time.sleep(_sleep_for(attempt))

    return None, last_error


def fetch_lightcurves_resumable(df_loci, label, cache_dir, max_workers=4, max_retries=6):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    locus_ids = df_loci["locus_id"].dropna().astype(str).drop_duplicates().tolist()
    frames = []
    failures = []

    print(f"  Fetching {len(locus_ids):,} lightcurves with {max_workers} workers")

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {
            pool.submit(_fetch_one_lightcurve_cached, lid, label, cache_dir, max_retries): lid
            for lid in locus_ids
        }

        for i, future in enumerate(as_completed(futures), start=1):
            lid = futures[future]
            frame, err = future.result()
            if err:
                failures.append({"locus_id": lid, "error": err})
            elif frame is not None:
                frames.append(frame)

            if i % 100 == 0 or i == len(locus_ids):
                print(f"    {i}/{len(locus_ids)} lightcurves done; failures={len(failures)}")

    alerts = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    return alerts, pd.DataFrame(failures)


def ingest_full_night_no_compromise(
    data_root,
    mjd_min,
    mjd_max,
    lsst_only=True,
    overwrite_mismatched=True,
    max_results_per_chunk=100,
    split_threshold=95,
    min_chunk_seconds=30.0,
    min_ra_degrees=0.05,
    max_retries=6,
    lightcurve_workers=4,
):
    data_root = Path(data_root)
    date_utc = history.mjd_to_utc_date(mjd_min)
    label = f"Full History {history.display_date(date_utc)}"
    paths = history.nightly_paths(data_root, date_utc)

    existing = history.read_manifest(data_root, date_utc)
    if existing:
        same_window = (
            abs(float(existing.get("mjd_min", -1)) - float(mjd_min)) < 1e-9
            and abs(float(existing.get("mjd_max", -1)) - float(mjd_max)) < 1e-9
        )
        has_lightcurves = int(existing.get("alert_rows", 0)) > 0
        if same_window and has_lightcurves and existing.get("status") == "complete":
            print(f"  Resume: complete full night with lightcurves already exists for {date_utc}")
            return existing
        if overwrite_mismatched:
            print(f"  Existing partition for {date_utc} is incomplete/mismatched; overwriting it.")

    started = _now_utc()
    t0 = time.time()

    print(f"\nFULL NIGHT {date_utc}  MJD [{mjd_min}, {mjd_max}]")
    print("  Stage 1: exhaustive LSST-only locus extraction")

    df_raw, report, saturated_leaf_count = full_loci_for_night(
        data_root=data_root,
        date_utc=date_utc,
        mjd_min=mjd_min,
        mjd_max=mjd_max,
        lsst_only=lsst_only,
        max_results_per_chunk=max_results_per_chunk,
        split_threshold=split_threshold,
        min_chunk_seconds=min_chunk_seconds,
        min_ra_degrees=min_ra_degrees,
        max_retries=max_retries,
    )

    ingested_at = _now_utc()
    df_loci = history.prepare_loci(df_raw, date_utc, mjd_min, mjd_max, ingested_at)
    paths["dir"].mkdir(parents=True, exist_ok=True)
    df_loci.to_parquet(paths["loci"], index=False)
    report.to_parquet(paths["dir"] / "chunk_report.parquet", index=False)

    if saturated_leaf_count:
        raise RuntimeError(
            f"{date_utc} still has {saturated_leaf_count} saturated final chunks. "
            "Lower min_ra_degrees before accepting this as complete."
        )

    print("  Stage 2: full lightcurve extraction")
    df_alerts_raw, lc_failures = fetch_lightcurves_resumable(
        df_loci,
        label,
        Path(data_root) / "cache" / "lightcurves" / date_utc,
        max_workers=lightcurve_workers,
        max_retries=max_retries,
    )

    if not lc_failures.empty:
        lc_failures.to_parquet(paths["dir"] / "lightcurve_failures.parquet", index=False)
        raise RuntimeError(
            f"{date_utc} has {len(lc_failures)} lightcurve failures after retries. "
            "Rerun the same cell; cached successes will be reused."
        )

    df_alerts = history.prepare_alerts(df_alerts_raw, date_utc, label)
    df_alerts.to_parquet(paths["alerts"], index=False)

    validation = history.validation_summary(
        df_loci,
        df_alerts,
        mjd_min=mjd_min,
        mjd_max=mjd_max,
        lsst_only=lsst_only,
    )
    counts = query.lsst_identifier_counts(df_loci)

    manifest = {
        "date_utc": date_utc,
        "mjd_min": float(mjd_min),
        "mjd_max": float(mjd_max),
        "query_tag": None,
        "target_loci": None,
        "actual_loci": int(len(df_loci)),
        "alert_rows": int(len(df_alerts)),
        "chunk_count": int((report["status"].str.startswith("accepted")).sum()),
        "split_count": int((report["status"].str.startswith("split")).sum()),
        "saturated_chunk_count": int(saturated_leaf_count),
        "status": "complete",
        "survey_mode": "lsst",
        "lsst_filter_used": bool(lsst_only),
        "lsst_filter": query.lsst_identifier_filter() if lsst_only else None,
        "parallel_shards": 1,
        "lsst_dia_count": counts["lsst_dia_count"],
        "lsst_ss_count": counts["lsst_ss_count"],
        "ztf_object_id_count": counts["ztf_object_id_count"],
        "started_at_utc": started,
        "finished_at_utc": _now_utc(),
        "runtime_seconds": round(time.time() - t0, 2),
        "validation": validation,
        "paths": {
            "loci": str(paths["loci"]),
            "alerts": str(paths["alerts"]),
            "manifest": str(paths["manifest"]),
        },
    }

    with open(paths["manifest"], "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, sort_keys=True)
        f.write("\n")

    history.update_cumulative_indexes(data_root)
    print(f"  COMPLETE {date_utc}: {len(df_loci):,} loci, {len(df_alerts):,} alert rows")
    return manifest

In [23]:
from pathlib import Path
from src import history

DATA_ROOT = Path("/home/mdarim/ANTARES_Analysis_Data")

manifests = []
failures = []

for date_utc, lo, hi in history.iter_night_windows(61095.0, 61100.0):
    try:
        manifest = ingest_full_night_no_compromise(
            data_root=DATA_ROOT,
            mjd_min=lo,
            mjd_max=hi,
            lsst_only=True,
            overwrite_mismatched=True,
            max_results_per_chunk=100,
            split_threshold=95,
            min_chunk_seconds=30.0,
            min_ra_degrees=0.05,
            max_retries=6,
            lightcurve_workers=4,
        )
        manifests.append(manifest)
    except Exception as exc:
        print(f"\nFAILED {date_utc}: {exc}")
        failures.append((date_utc, lo, hi, str(exc)))
        break

loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

print("Finished nights:", len(manifests))
print("Failures:", failures)
print("Cumulative loci rows:", len(loci_index))
display(nightly_summary)

  Existing partition for 2026-02-24 is incomplete/mismatched; overwriting it.

FULL NIGHT 2026-02-24  MJD [61095.0, 61096.0]
  Stage 1: exhaustive LSST-only locus extraction
        1. MJD 61095.000000-61096.000000 RA    0.00- 360.00   100 loci split_time (live; 2 queued)
        2. MJD 61095.000000-61095.500000 RA    0.00- 360.00   100 loci split_time (live; 3 queued)
        3. MJD 61095.000000-61095.250000 RA    0.00- 360.00   100 loci split_time (live; 4 queued)
        4. MJD 61095.000000-61095.125000 RA    0.00- 360.00   100 loci split_time (live; 5 queued)
        5. MJD 61095.000000-61095.062500 RA    0.00- 360.00   100 loci split_time (live; 6 queued)
        6. MJD 61095.000000-61095.031250 RA    0.00- 360.00   100 loci split_time (live; 7 queued)
        7. MJD 61095.000000-61095.015625 RA    0.00- 360.00     0 loci accepted (live; 6 queued)
        8. MJD 61095.015625-61095.031250 RA    0.00- 360.00   100 loci split_time (live; 7 queued)
        9. MJD 61095.015625-61095.02

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



    16161. MJD 61095.220703-61095.220947 RA  149.06- 150.47    88 loci accepted (live; 12 queued)
    16162. MJD 61095.220703-61095.220947 RA  150.47- 151.88    64 loci accepted (live; 11 queued)
    16163. MJD 61095.220703-61095.220947 RA  151.88- 157.50     0 loci accepted (live; 10 queued)
    16164. MJD 61095.220703-61095.220947 RA  157.50- 180.00     0 loci accepted (live; 9 queued)
    16165. MJD 61095.220703-61095.220947 RA  180.00- 360.00     0 loci accepted (live; 8 queued)
    16166. MJD 61095.220947-61095.221191 RA    0.00- 360.00     0 loci accepted (live; 7 queued)
    16167. MJD 61095.221191-61095.221680 RA    0.00- 360.00   100 loci split_time (live; 8 queued)
    16168. MJD 61095.221191-61095.221436 RA    0.00- 360.00   100 loci split_ra (live; 9 queued)
    16169. MJD 61095.221191-61095.221436 RA    0.00- 180.00   100 loci split_ra (live; 10 queued)
    16170. MJD 61095.221191-61095.221436 RA    0.00-  90.00     0 loci accepted (live; 9 queued)
    16171. MJD 61095.221

,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,lsst_ss_count,ztf_object_id_count,lsst_only_pass,history_start_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
0,2026-02-24,2026/2/24,61095.0,61096.00,None,1000,1000,0,27,33,...,0,0,True,True,37.20,2026-05-24T09:02:52+00:00,2026-05-24T09:03:29+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
1,2026-02-25,2026/2/25,61096.0,61096.25,None,1000,1000,0,6,8,...,0,0,True,True,41.50,2026-05-23T22:05:01+00:00,2026-05-23T22:05:42+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
2,2026-02-26,2026/2/26,61097.0,61098.00,None,1000,1000,0,20,25,...,0,0,True,True,37.47,2026-05-24T09:03:29+00:00,2026-05-24T09:04:06+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
3,2026-02-27,2026/2/27,61098.0,61099.00,None,1000,1000,0,22,29,...,0,0,True,True,122.44,2026-05-24T09:04:07+00:00,2026-05-24T09:06:09+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
4,2026-02-28,2026/2/28,61099.0,61100.00,None,1000,1000,0,33,39,...,284,3,True,True,49.32,2026-05-24T09:06:09+00:00,2026-05-24T09:06:58+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...


In [24]:
from pathlib import Path
import json
import time
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from antares_client.search import search as antares_search, get_by_id

from src import history, query


# ============================================================
# USER SETTINGS: CHANGE THESE FIRST
# ============================================================
DATA_ROOT = Path("/home/mdarim/ANTARES_Analysis_Data")

# One full night. Change these two values for a different night.
MJD_START = 61096.0
MJD_STOP = 61097.0

FETCH_LIGHTCURVES = True

# Query geometry. Start here; tune only if needed.
TIME_BIN_MINUTES = 10
RA_BINS = 24
DEC_BINS = 6

# Moderate cap: faster than 100, safer than 10000.
MAX_RESULTS_PER_TILE = 500
SATURATION_THRESHOLD = 475

# Retry/cache behavior.
MAX_RETRIES = 6
LIGHTCURVE_WORKERS = 4

# Final safety floors. If a tile is still saturated below these,
# the night is not accepted as complete.
MIN_TIME_SECONDS = 30
MIN_RA_DEGREES = 0.05
MIN_DEC_DEGREES = 0.05


# ============================================================
# INTERNAL HELPERS
# ============================================================
def now_utc():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def retry_sleep(attempt):
    return [20, 60, 120, 240, 480][min(attempt - 1, 4)]


def frange(start, stop, step):
    x = float(start)
    while x < float(stop):
        y = min(x + step, float(stop))
        yield x, y
        x = y


def query_tile_once(mjd_min, mjd_max, ra_min, ra_max, dec_min, dec_max, include_upper):
    upper_op = "lte" if include_upper else "lt"

    q = {
        "query": {
            "bool": {
                "filter": [
                    {
                        "range": {
                            "properties.newest_alert_observation_time": {
                                "gte": float(mjd_min),
                                upper_op: float(mjd_max),
                            }
                        }
                    },
                    {"range": {"ra": {"gte": float(ra_min), "lt": float(ra_max)}}},
                    {"range": {"dec": {"gte": float(dec_min), "lt": float(dec_max)}}},
                    query.lsst_identifier_filter(),
                ]
            }
        }
    }

    rows = []
    for locus in antares_search(q):
        rows.append(query.locus_to_record(locus))
        if len(rows) >= MAX_RESULTS_PER_TILE:
            break

    return pd.DataFrame(rows)


def query_tile_with_retry(*args):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return query_tile_once(*args)
        except Exception as exc:
            last_error = exc
            print(f"      query failed attempt {attempt}/{MAX_RETRIES}: {exc}")
            if attempt < MAX_RETRIES:
                wait = retry_sleep(attempt)
                print(f"      sleeping {wait}s")
                time.sleep(wait)
    raise last_error


def tile_cache_path(cache_dir, tile):
    mjd0, mjd1, ra0, ra1, dec0, dec1, include_upper = tile
    name = (
        f"mjd_{mjd0:.8f}_{mjd1:.8f}_"
        f"ra_{ra0:.4f}_{ra1:.4f}_"
        f"dec_{dec0:.4f}_{dec1:.4f}_"
        f"upper_{int(include_upper)}.parquet"
    ).replace(".", "p").replace("-", "m")
    return cache_dir / name


def split_tile(tile):
    mjd0, mjd1, ra0, ra1, dec0, dec1, include_upper = tile
    time_seconds = (mjd1 - mjd0) * 86400.0
    ra_width = ra1 - ra0
    dec_width = dec1 - dec0

    # Prefer sky splitting before going too tiny in time.
    if dec_width > MIN_DEC_DEGREES * 1.01:
        mid = (dec0 + dec1) / 2.0
        return [
            (mjd0, mjd1, ra0, ra1, dec0, mid, include_upper),
            (mjd0, mjd1, ra0, ra1, mid, dec1, include_upper),
        ]

    if ra_width > MIN_RA_DEGREES * 1.01:
        mid = (ra0 + ra1) / 2.0
        return [
            (mjd0, mjd1, ra0, mid, dec0, dec1, include_upper),
            (mjd0, mjd1, mid, ra1, dec0, dec1, include_upper),
        ]

    if time_seconds > MIN_TIME_SECONDS * 1.01:
        mid = (mjd0 + mjd1) / 2.0
        return [
            (mjd0, mid, ra0, ra1, dec0, dec1, False),
            (mid, mjd1, ra0, ra1, dec0, dec1, include_upper),
        ]

    return []


def extract_full_loci_for_night(data_root, mjd_start, mjd_stop):
    date_utc = history.mjd_to_utc_date(mjd_start)
    cache_dir = data_root / "cache" / "tile_loci" / date_utc
    cache_dir.mkdir(parents=True, exist_ok=True)

    time_step = TIME_BIN_MINUTES / 1440.0
    ra_step = 360.0 / RA_BINS
    dec_step = 180.0 / DEC_BINS

    pending = []
    for mjd0, mjd1 in frange(mjd_start, mjd_stop, time_step):
        include_upper = mjd1 >= mjd_stop
        for ra0, ra1 in frange(0.0, 360.0, ra_step):
            for dec0, dec1 in frange(-90.0, 90.0, dec_step):
                pending.append((mjd0, mjd1, ra0, ra1, dec0, dec1, include_upper))

    accepted_paths = []
    report_rows = []
    saturated_final = []
    n_done = 0

    print(f"Initial tiles: {len(pending):,}")

    while pending:
        tile = pending.pop(0)
        mjd0, mjd1, ra0, ra1, dec0, dec1, include_upper = tile
        path = tile_cache_path(cache_dir, tile)

        if path.exists():
            df = pd.read_parquet(path)
            source = "cache"
        else:
            df = query_tile_with_retry(mjd0, mjd1, ra0, ra1, dec0, dec1, include_upper)
            source = "live"

        n = len(df)
        saturated = n >= SATURATION_THRESHOLD

        if saturated:
            children = split_tile(tile)
            if children:
                pending = children + pending
                status = "split"
            else:
                df.to_parquet(path, index=False)
                accepted_paths.append(path)
                saturated_final.append(tile)
                status = "accepted_saturated_final"
        else:
            df.to_parquet(path, index=False)
            accepted_paths.append(path)
            status = "accepted"

        n_done += 1
        report_rows.append({
            "mjd_min": mjd0,
            "mjd_max": mjd1,
            "ra_min": ra0,
            "ra_max": ra1,
            "dec_min": dec0,
            "dec_max": dec1,
            "include_upper": include_upper,
            "n_loci": n,
            "status": status,
            "source": source,
            "remaining": len(pending),
        })

        if n_done % 100 == 0 or saturated or status.startswith("accepted_saturated"):
            print(
                f"{n_done:>6} done | {len(pending):>6} queued | "
                f"MJD {mjd0:.6f}-{mjd1:.6f} "
                f"RA {ra0:6.1f}-{ra1:6.1f} "
                f"Dec {dec0:6.1f}-{dec1:6.1f} "
                f"{n:>4} loci {status} ({source})"
            )

    frames = [pd.read_parquet(p) for p in accepted_paths]
    loci = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    if not loci.empty and "locus_id" in loci.columns:
        loci = loci.drop_duplicates("locus_id", keep="last").reset_index(drop=True)

    report = pd.DataFrame(report_rows)
    return loci, report, saturated_final


def fetch_one_lc_cached(locus_id, label, cache_dir):
    path = cache_dir / f"{locus_id}.parquet"
    if path.exists():
        return pd.read_parquet(path), None

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            locus = get_by_id(locus_id)
            lc = locus.lightcurve
            if lc is None or lc.empty:
                lc = pd.DataFrame({
                    "locus_id": [locus_id],
                    "range_label": [label],
                    "empty_lightcurve": [True],
                })
            else:
                lc = lc.copy()
                lc["locus_id"] = locus_id
                lc["range_label"] = label

            lc.to_parquet(path, index=False)
            return lc, None

        except Exception as exc:
            last_error = str(exc)
            if attempt < MAX_RETRIES:
                time.sleep(retry_sleep(attempt))

    return None, last_error


def fetch_all_lightcurves(df_loci, date_utc, label, data_root):
    cache_dir = data_root / "cache" / "lightcurves" / date_utc
    cache_dir.mkdir(parents=True, exist_ok=True)

    locus_ids = df_loci["locus_id"].dropna().astype(str).drop_duplicates().tolist()
    frames = []
    failures = []

    print(f"Fetching {len(locus_ids):,} lightcurves with {LIGHTCURVE_WORKERS} workers")

    with ThreadPoolExecutor(max_workers=LIGHTCURVE_WORKERS) as pool:
        futures = {
            pool.submit(fetch_one_lc_cached, lid, label, cache_dir): lid
            for lid in locus_ids
        }

        for i, future in enumerate(as_completed(futures), start=1):
            lid = futures[future]
            frame, err = future.result()
            if err:
                failures.append({"locus_id": lid, "error": err})
            elif frame is not None:
                frames.append(frame)

            if i % 100 == 0 or i == len(locus_ids):
                print(f"  {i}/{len(locus_ids)} lightcurves complete; failures={len(failures)}")

    alerts = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    return alerts, pd.DataFrame(failures)


# ============================================================
# RUN ONE FULL NIGHT
# ============================================================
DATA_ROOT.mkdir(parents=True, exist_ok=True)

date_utc = history.mjd_to_utc_date(MJD_START)
label = f"Full History {history.display_date(date_utc)}"
paths = history.nightly_paths(DATA_ROOT, date_utc)
paths["dir"].mkdir(parents=True, exist_ok=True)

print(f"FULL NIGHT: {date_utc}")
print(f"MJD range : [{MJD_START}, {MJD_STOP}]")
print(f"Data root : {DATA_ROOT}")

started = now_utc()
t0 = time.time()

print("\nStage 1: full LSST-only locus extraction")
df_loci_raw, tile_report, saturated_final = extract_full_loci_for_night(
    DATA_ROOT,
    MJD_START,
    MJD_STOP,
)

tile_report.to_parquet(paths["dir"] / "tile_report.parquet", index=False)

if saturated_final:
    print(f"\nNOT COMPLETE: {len(saturated_final)} final saturated tiles remain.")
    print("Lower MAX_RESULTS_PER_TILE risk is not the fix; lower MIN_RA_DEGREES/MIN_DEC_DEGREES or increase initial sky bins.")
    raise RuntimeError(f"{date_utc} has saturated final tiles; not accepting as complete.")

df_loci = history.prepare_loci(df_loci_raw, date_utc, MJD_START, MJD_STOP, now_utc())
df_loci.to_parquet(paths["loci"], index=False)

print(f"Loci complete: {len(df_loci):,}")

if FETCH_LIGHTCURVES:
    print("\nStage 2: full lightcurve extraction")
    df_alerts_raw, lc_failures = fetch_all_lightcurves(df_loci, date_utc, label, DATA_ROOT)

    if not lc_failures.empty:
        lc_failures.to_parquet(paths["dir"] / "lightcurve_failures.parquet", index=False)
        raise RuntimeError(f"{len(lc_failures)} lightcurves failed after retries. Rerun this cell to resume.")

    df_alerts = history.prepare_alerts(df_alerts_raw, date_utc, label)
else:
    df_alerts = history.prepare_alerts(pd.DataFrame(), date_utc, label)

df_alerts.to_parquet(paths["alerts"], index=False)

validation = history.validation_summary(
    df_loci,
    df_alerts,
    mjd_min=MJD_START,
    mjd_max=MJD_STOP,
    lsst_only=True,
)

counts = query.lsst_identifier_counts(df_loci)

manifest = {
    "date_utc": date_utc,
    "mjd_min": float(MJD_START),
    "mjd_max": float(MJD_STOP),
    "query_tag": None,
    "target_loci": None,
    "actual_loci": int(len(df_loci)),
    "alert_rows": int(len(df_alerts)),
    "chunk_count": int((tile_report["status"].str.startswith("accepted")).sum()),
    "split_count": int((tile_report["status"] == "split").sum()),
    "saturated_chunk_count": 0,
    "status": "complete",
    "survey_mode": "lsst",
    "lsst_filter_used": True,
    "lsst_filter": query.lsst_identifier_filter(),
    "parallel_shards": 1,
    "lsst_dia_count": counts["lsst_dia_count"],
    "lsst_ss_count": counts["lsst_ss_count"],
    "ztf_object_id_count": counts["ztf_object_id_count"],
    "started_at_utc": started,
    "finished_at_utc": now_utc(),
    "runtime_seconds": round(time.time() - t0, 2),
    "validation": validation,
    "paths": {
        "loci": str(paths["loci"]),
        "alerts": str(paths["alerts"]),
        "manifest": str(paths["manifest"]),
    },
}

with open(paths["manifest"], "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, sort_keys=True)
    f.write("\n")

loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

print("\nCOMPLETE")
print("Date:", date_utc)
print("Loci:", len(df_loci))
print("Alert/lightcurve rows:", len(df_alerts))
print("Runtime seconds:", manifest["runtime_seconds"])
display(nightly_summary.tail())

FULL NIGHT: 2026-02-25
MJD range : [61096.0, 61097.0]
Data root : /home/mdarim/ANTARES_Analysis_Data

Stage 1: full LSST-only locus extraction
Initial tiles: 20,736
   100 done |  20636 queued | MJD 61096.000000-61096.006944 RA  240.0- 255.0 Dec    0.0-  30.0    0 loci accepted (live)
   200 done |  20536 queued | MJD 61096.006944-61096.013889 RA  135.0- 150.0 Dec  -60.0- -30.0    0 loci accepted (live)
   300 done |  20436 queued | MJD 61096.013889-61096.020833 RA   15.0-  30.0 Dec   60.0-  90.0    0 loci accepted (live)
   400 done |  20336 queued | MJD 61096.013889-61096.020833 RA  270.0- 285.0 Dec    0.0-  30.0    0 loci accepted (live)
   500 done |  20236 queued | MJD 61096.020833-61096.027778 RA  165.0- 180.0 Dec  -60.0- -30.0    0 loci accepted (live)
   596 done |  20142 queued | MJD 61096.027778-61096.034722 RA   45.0-  60.0 Dec  -60.0- -30.0  500 loci split (live)
   597 done |  20143 queued | MJD 61096.027778-61096.034722 RA   45.0-  60.0 Dec  -60.0- -45.0  500 loci split (

KeyboardInterrupt: 

In [74]:
from pathlib import Path
import json
import time
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from antares_client.search import search as antares_search, get_by_id

from src import history, query


# ============================================================
# USER SETTINGS
# ============================================================
DATA_ROOT = Path("/home/mdarim/ANTARES_Analysis_Data")

# Change these for the one full night you want.
MJD_START = 61114.0
MJD_STOP = 61115.0

FETCH_LIGHTCURVES = True

# Probe-first strategy:
# If a tile returns 50 rows, assume it may contain more and split.
PROBE_LIMIT = 50
PROBE_THRESHOLD = 50

# Coarser initial grid than before; dense tiles split automatically.
TIME_BIN_MINUTES = 30
RA_BINS = 24
DEC_BINS = 6

MAX_RETRIES = 6
LIGHTCURVE_WORKERS = 4

# Safety floors. If a tile is still full below these, we refuse to call the night complete.
MIN_TIME_SECONDS = 30
MIN_RA_DEGREES = 0.05
MIN_DEC_DEGREES = 0.05

# New cache version so it does not mix with the earlier 500-row tile run.
CACHE_VERSION = "probe50_v1"


# ============================================================
# HELPERS
# ============================================================
def now_utc():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def retry_sleep(attempt):
    return [20, 60, 120, 240, 480][min(attempt - 1, 4)]


def frange(start, stop, step):
    x = float(start)
    while x < float(stop) - 1e-12:
        y = min(x + step, float(stop))
        yield x, y
        x = y


def query_tile_once(mjd_min, mjd_max, ra_min, ra_max, dec_min, dec_max, include_upper):
    upper_op = "lte" if include_upper else "lt"

    q = {
        "query": {
            "bool": {
                "filter": [
                    {
                        "range": {
                            "properties.newest_alert_observation_time": {
                                "gte": float(mjd_min),
                                upper_op: float(mjd_max),
                            }
                        }
                    },
                    {"range": {"ra": {"gte": float(ra_min), "lt": float(ra_max)}}},
                    {"range": {"dec": {"gte": float(dec_min), "lt": float(dec_max)}}},
                    query.lsst_identifier_filter(),
                ]
            }
        }
    }

    rows = []
    for locus in antares_search(q):
        rows.append(query.locus_to_record(locus))
        if len(rows) >= PROBE_LIMIT:
            break

    return pd.DataFrame(rows)


def query_tile_with_retry(*args):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return query_tile_once(*args)
        except Exception as exc:
            last_error = exc
            print(f"      query failed attempt {attempt}/{MAX_RETRIES}: {exc}")
            if attempt < MAX_RETRIES:
                wait = retry_sleep(attempt)
                print(f"      sleeping {wait}s")
                time.sleep(wait)
    raise last_error


def tile_cache_path(cache_dir, tile):
    mjd0, mjd1, ra0, ra1, dec0, dec1, include_upper = tile
    name = (
        f"mjd_{mjd0:.8f}_{mjd1:.8f}_"
        f"ra_{ra0:.4f}_{ra1:.4f}_"
        f"dec_{dec0:.4f}_{dec1:.4f}_"
        f"upper_{int(include_upper)}.parquet"
    ).replace(".", "p").replace("-", "m")
    return cache_dir / name


def split_tile(tile):
    mjd0, mjd1, ra0, ra1, dec0, dec1, include_upper = tile
    time_seconds = (mjd1 - mjd0) * 86400.0
    ra_width = ra1 - ra0
    dec_width = dec1 - dec0

    # Sky-first splitting avoids deep pagination in dense observing strips.
    if dec_width > MIN_DEC_DEGREES * 1.01:
        mid = (dec0 + dec1) / 2.0
        return [
            (mjd0, mjd1, ra0, ra1, dec0, mid, include_upper),
            (mjd0, mjd1, ra0, ra1, mid, dec1, include_upper),
        ]

    if ra_width > MIN_RA_DEGREES * 1.01:
        mid = (ra0 + ra1) / 2.0
        return [
            (mjd0, mjd1, ra0, mid, dec0, dec1, include_upper),
            (mjd0, mjd1, mid, ra1, dec0, dec1, include_upper),
        ]

    if time_seconds > MIN_TIME_SECONDS * 1.01:
        mid = (mjd0 + mjd1) / 2.0
        return [
            (mjd0, mid, ra0, ra1, dec0, dec1, False),
            (mid, mjd1, ra0, ra1, dec0, dec1, include_upper),
        ]

    return []


def initial_tiles(mjd_start, mjd_stop):
    time_step = TIME_BIN_MINUTES / 1440.0
    ra_step = 360.0 / RA_BINS
    dec_step = 180.0 / DEC_BINS

    tiles = []
    for mjd0, mjd1 in frange(mjd_start, mjd_stop, time_step):
        include_upper = mjd1 >= mjd_stop
        for ra0, ra1 in frange(0.0, 360.0, ra_step):
            for dec0, dec1 in frange(-90.0, 90.0, dec_step):
                tiles.append((mjd0, mjd1, ra0, ra1, dec0, dec1, include_upper))
    return tiles


def extract_full_loci_probe_first(data_root, mjd_start, mjd_stop):
    date_utc = history.mjd_to_utc_date(mjd_start)
    cache_dir = data_root / "cache" / CACHE_VERSION / "tile_loci" / date_utc
    cache_dir.mkdir(parents=True, exist_ok=True)

    pending = initial_tiles(mjd_start, mjd_stop)
    accepted_paths = []
    report_rows = []
    saturated_final = []
    done = 0
    t0 = time.time()

    print(f"Initial tiles: {len(pending):,}")
    print(f"Probe limit: {PROBE_LIMIT}; split if n >= {PROBE_THRESHOLD}")

    while pending:
        tile = pending.pop(0)
        mjd0, mjd1, ra0, ra1, dec0, dec1, include_upper = tile
        path = tile_cache_path(cache_dir, tile)

        if path.exists():
            df = pd.read_parquet(path)
            source = "cache"
        else:
            df = query_tile_with_retry(mjd0, mjd1, ra0, ra1, dec0, dec1, include_upper)
            source = "live"

        n = len(df)
        saturated = n >= PROBE_THRESHOLD

        if saturated:
            children = split_tile(tile)
            if children:
                pending = children + pending
                status = "split"
            else:
                df.to_parquet(path, index=False)
                accepted_paths.append(path)
                saturated_final.append(tile)
                status = "accepted_saturated_final"
        else:
            df.to_parquet(path, index=False)
            accepted_paths.append(path)
            status = "accepted"

        done += 1
        report_rows.append({
            "mjd_min": mjd0,
            "mjd_max": mjd1,
            "ra_min": ra0,
            "ra_max": ra1,
            "dec_min": dec0,
            "dec_max": dec1,
            "include_upper": include_upper,
            "n_loci": n,
            "status": status,
            "source": source,
            "remaining": len(pending),
        })

        if done % 250 == 0 or saturated or status == "accepted_saturated_final":
            elapsed = time.time() - t0
            rate = done / elapsed if elapsed else 0
            print(
                f"{done:>6} done | {len(pending):>6} queued | "
                f"{rate:5.1f} tiles/s | "
                f"MJD {mjd0:.6f}-{mjd1:.6f} "
                f"RA {ra0:6.1f}-{ra1:6.1f} "
                f"Dec {dec0:6.1f}-{dec1:6.1f} "
                f"{n:>3} loci {status} ({source})"
            )

    frames = [pd.read_parquet(p) for p in accepted_paths]
    loci = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

    if not loci.empty and "locus_id" in loci.columns:
        loci = loci.drop_duplicates("locus_id", keep="last").reset_index(drop=True)

    return loci, pd.DataFrame(report_rows), saturated_final


def fetch_one_lightcurve_cached(locus_id, label, cache_dir):
    path = cache_dir / f"{locus_id}.parquet"
    if path.exists():
        return pd.read_parquet(path), None

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            locus = get_by_id(locus_id)
            lc = locus.lightcurve
            if lc is None or lc.empty:
                lc = pd.DataFrame({
                    "locus_id": [locus_id],
                    "range_label": [label],
                    "empty_lightcurve": [True],
                })
            else:
                lc = lc.copy()
                lc["locus_id"] = locus_id
                lc["range_label"] = label

            lc.to_parquet(path, index=False)
            return lc, None

        except Exception as exc:
            last_error = str(exc)
            if attempt < MAX_RETRIES:
                time.sleep(retry_sleep(attempt))

    return None, last_error


def fetch_all_lightcurves_cached(df_loci, date_utc, label, data_root):
    cache_dir = data_root / "cache" / CACHE_VERSION / "lightcurves" / date_utc
    cache_dir.mkdir(parents=True, exist_ok=True)

    locus_ids = df_loci["locus_id"].dropna().astype(str).drop_duplicates().tolist()
    frames = []
    failures = []

    print(f"Fetching {len(locus_ids):,} lightcurves with {LIGHTCURVE_WORKERS} workers")

    with ThreadPoolExecutor(max_workers=LIGHTCURVE_WORKERS) as pool:
        futures = {
            pool.submit(fetch_one_lightcurve_cached, lid, label, cache_dir): lid
            for lid in locus_ids
        }

        for i, future in enumerate(as_completed(futures), start=1):
            lid = futures[future]
            frame, err = future.result()

            if err:
                failures.append({"locus_id": lid, "error": err})
            elif frame is not None:
                frames.append(frame)

            if i % 100 == 0 or i == len(locus_ids):
                print(f"  {i}/{len(locus_ids)} lightcurves complete; failures={len(failures)}")

    alerts = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    return alerts, pd.DataFrame(failures)


# ============================================================
# RUN ONE FULL NIGHT
# ============================================================
DATA_ROOT.mkdir(parents=True, exist_ok=True)

date_utc = history.mjd_to_utc_date(MJD_START)
label = f"Full History {history.display_date(date_utc)}"
paths = history.nightly_paths(DATA_ROOT, date_utc)
paths["dir"].mkdir(parents=True, exist_ok=True)

print(f"FULL NIGHT: {date_utc}")
print(f"MJD range : [{MJD_START}, {MJD_STOP}]")
print(f"Data root : {DATA_ROOT}")
print(f"Cache     : {CACHE_VERSION}")

started = now_utc()
t0 = time.time()

print("\nStage 1: probe-first full LSST-only locus extraction")
df_loci_raw, tile_report, saturated_final = extract_full_loci_probe_first(
    DATA_ROOT,
    MJD_START,
    MJD_STOP,
)

tile_report.to_parquet(paths["dir"] / f"tile_report_{CACHE_VERSION}.parquet", index=False)

if saturated_final:
    print(f"\nNOT COMPLETE: {len(saturated_final)} final saturated tiles remain.")
    print("Increase initial RA/Dec bins or lower MIN_RA_DEGREES/MIN_DEC_DEGREES, then rerun.")
    raise RuntimeError(f"{date_utc} has saturated final tiles; not accepting as complete.")

df_loci = history.prepare_loci(df_loci_raw, date_utc, MJD_START, MJD_STOP, now_utc())
df_loci.to_parquet(paths["loci"], index=False)

print(f"\nLoci complete: {len(df_loci):,}")

if FETCH_LIGHTCURVES:
    print("\nStage 2: full lightcurve extraction")
    df_alerts_raw, lc_failures = fetch_all_lightcurves_cached(
        df_loci,
        date_utc,
        label,
        DATA_ROOT,
    )

    if not lc_failures.empty:
        lc_failures.to_parquet(paths["dir"] / f"lightcurve_failures_{CACHE_VERSION}.parquet", index=False)
        raise RuntimeError(f"{len(lc_failures)} lightcurves failed after retries. Rerun this cell to resume.")

    df_alerts = history.prepare_alerts(df_alerts_raw, date_utc, label)
else:
    df_alerts = history.prepare_alerts(pd.DataFrame(), date_utc, label)

df_alerts.to_parquet(paths["alerts"], index=False)

validation = history.validation_summary(
    df_loci,
    df_alerts,
    mjd_min=MJD_START,
    mjd_max=MJD_STOP,
    lsst_only=True,
)

counts = query.lsst_identifier_counts(df_loci)

manifest = {
    "date_utc": date_utc,
    "mjd_min": float(MJD_START),
    "mjd_max": float(MJD_STOP),
    "query_tag": None,
    "target_loci": None,
    "actual_loci": int(len(df_loci)),
    "alert_rows": int(len(df_alerts)),
    "chunk_count": int((tile_report["status"].str.startswith("accepted")).sum()),
    "split_count": int((tile_report["status"] == "split").sum()),
    "saturated_chunk_count": 0,
    "status": "complete",
    "survey_mode": "lsst",
    "lsst_filter_used": True,
    "lsst_filter": query.lsst_identifier_filter(),
    "parallel_shards": 1,
    "lsst_dia_count": counts["lsst_dia_count"],
    "lsst_ss_count": counts["lsst_ss_count"],
    "ztf_object_id_count": counts["ztf_object_id_count"],
    "started_at_utc": started,
    "finished_at_utc": now_utc(),
    "runtime_seconds": round(time.time() - t0, 2),
    "validation": validation,
    "paths": {
        "loci": str(paths["loci"]),
        "alerts": str(paths["alerts"]),
        "manifest": str(paths["manifest"]),
    },
    "cache_version": CACHE_VERSION,
    "probe_limit": PROBE_LIMIT,
    "time_bin_minutes": TIME_BIN_MINUTES,
    "ra_bins": RA_BINS,
    "dec_bins": DEC_BINS,
}

with open(paths["manifest"], "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, sort_keys=True)
    f.write("\n")

loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

print("\nCOMPLETE")
print("Date:", date_utc)
print("Loci:", len(df_loci))
print("Alert/lightcurve rows:", len(df_alerts))
print("Runtime seconds:", manifest["runtime_seconds"])
display(nightly_summary.tail())

FULL NIGHT: 2026-03-15
MJD range : [61114.0, 61115.0]
Data root : /home/mdarim/ANTARES_Analysis_Data
Cache     : probe50_v1

Stage 1: probe-first full LSST-only locus extraction
Initial tiles: 6,912
Probe limit: 50; split if n >= 50
   250 done |   6662 queued |  17.4 tiles/s | MJD 61114.020833-61114.041667 RA  255.0- 270.0 Dec    0.0-  30.0   0 loci accepted (live)
   500 done |   6412 queued |  17.6 tiles/s | MJD 61114.062500-61114.083333 RA  165.0- 180.0 Dec  -60.0- -30.0   0 loci accepted (live)
   750 done |   6162 queued |  17.5 tiles/s | MJD 61114.104167-61114.125000 RA   60.0-  75.0 Dec   60.0-  90.0   0 loci accepted (live)
  1000 done |   5912 queued |  17.5 tiles/s | MJD 61114.125000-61114.145833 RA  330.0- 345.0 Dec    0.0-  30.0   0 loci accepted (live)
      query failed attempt 1/6: HTTPSConnectionPool(host='api.antares.noirlab.edu', port=443): Max retries exceeded with url: /v1/loci?sort=-properties.newest_alert_observation_time&elasticsearch_query%5Blocus_listing%5D=%7

,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,lsst_ss_count,ztf_object_id_count,lsst_only_pass,history_start_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
16,2026-03-13,2026/3/13,61112.0,61113.0,None,None,5,438,6912,0,...,5,5,True,True,949.95,2026-05-29T00:09:54+00:00,2026-05-29T00:25:44+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
17,2026-03-14,2026/3/14,61113.0,61114.0,None,None,2,82,6912,0,...,2,2,True,True,926.11,2026-05-29T00:43:16+00:00,2026-05-29T00:58:43+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
18,2026-03-15,2026/3/15,61114.0,61115.0,None,None,10,2818,6912,0,...,10,10,True,True,999.27,2026-05-29T01:15:36+00:00,2026-05-29T01:32:15+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
19,2026-05-25,2026/5/25,61185.0,61186.0,None,None,49926,1527986,9625,2713,...,49926,49926,True,True,199.40,2026-05-26T18:25:29+00:00,2026-05-26T18:28:48+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
20,2026-05-27,2026/5/27,61187.0,61188.0,None,None,25092,254055,8279,1367,...,25092,25092,True,True,123.16,2026-05-28T15:11:27+00:00,2026-05-28T15:13:30+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...


## 4. Nightly Update

This ingests the newest night, compares it against prior cumulative history, and appends it only after validation passes.


In [35]:
RUN_NIGHTLY_UPDATE = True

if RUN_NIGHTLY_UPDATE:
    nightly_result = history.run_nightly_update(
        data_root=DATA_ROOT,
        mjd_min=config.MJD1_MIN,
        mjd_max=config.MJD1_MAX,
        target_loci=config.HISTORY_TARGET_LOCI,
        fetch_lightcurves=config.HISTORY_FETCH_ALL_LIGHTCURVES,
        resume=True,
    )
    print(nightly_result['comparison'])
    print(nightly_result['manifest']['status'])


  Parent shards: 3 non-overlapping workers
  Chunked query 'Last Night 2026/5/21 shard 1/3'  MJD [61181.000000, 61181.333333]
    ES limit=10,000, split at >= 9,500, minimum chunk=30s
  Chunked query 'Last Night 2026/5/21 shard 2/3'  MJD [61181.333333, 61181.666667]
    ES limit=10,000, split at >= 9,500, minimum chunk=30s
  Chunked query 'Last Night 2026/5/21 shard 3/3'  MJD [61181.666667, 61182.000000]
    ES limit=10,000, split at >= 9,500, minimum chunk=30s
       1. 61181.666667-61182.000000   28800.0s       0 loci  accepted  (live; 0 queued)
  Last Night 2026/5/21 shard 3/3: 0 unique loci from 1 accepted chunks  (0 splits, 0.3s)
       1. 61181.333333-61181.666667   28800.0s       0 loci  accepted  (live; 0 queued)
  Last Night 2026/5/21 shard 2/3: 0 unique loci from 1 accepted chunks  (0 splits, 0.3s)
       1. 61181.000000-61181.333333   28800.0s      16 loci  accepted  (live; 0 queued)
  Last Night 2026/5/21 shard 1/3: 15 unique loci from 1 accepted chunks  (0 splits, 0.4s)
  

## 5. Inspect Stored Data


In [75]:
loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)
print(f'Cumulative index rows: {len(loci_index):,}')
display(nightly_summary.tail(20))


Cumulative index rows: 349,325


,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,lsst_ss_count,ztf_object_id_count,lsst_only_pass,history_start_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
1,2026-02-26,2026/2/26,61097.0,61098.0,None,None,96238,933004,11853,4941,...,96238,96238,True,True,24377.58,2026-05-25T09:23:14+00:00,2026-05-25T16:09:32+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
2,2026-02-27,2026/2/27,61098.0,61099.0,None,None,47909,1590484,8952,2040,...,47909,47909,True,True,11109.51,2026-05-25T17:46:27+00:00,2026-05-25T20:51:36+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
3,2026-02-28,2026/2/28,61099.0,61100.0,None,None,2919,139630,7017,105,...,2919,2919,True,True,1534.53,2026-05-26T00:14:24+00:00,2026-05-26T00:39:58+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
4,2026-03-01,2026/3/1,61100.0,61101.0,None,None,1361,138097,6958,46,...,1361,1361,True,True,1174.79,2026-05-26T00:44:59+00:00,2026-05-26T01:04:34+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
5,2026-03-02,2026/3/2,61101.0,61102.0,None,None,1531,87575,6970,58,...,1531,1531,True,True,1200.71,2026-05-26T01:12:01+00:00,2026-05-26T01:32:01+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
6,2026-03-03,2026/3/3,61102.0,61103.0,None,None,675,57283,6939,27,...,675,675,True,True,984.66,2026-05-26T01:37:27+00:00,2026-05-26T01:53:51+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
7,2026-03-04,2026/3/4,61103.0,61104.0,None,None,239,2530,6921,9,...,239,239,True,True,985.27,2026-05-28T05:54:42+00:00,2026-05-28T06:11:08+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
8,2026-03-05,2026/3/5,61104.0,61105.0,None,None,0,0,6912,0,...,0,0,False,True,973.44,2026-05-28T06:13:41+00:00,2026-05-28T06:29:55+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
9,2026-03-06,2026/3/6,61105.0,61106.0,None,None,2349,100660,6994,82,...,2349,2349,True,True,1484.29,2026-05-28T06:40:56+00:00,2026-05-28T07:05:40+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
10,2026-03-07,2026/3/7,61106.0,61107.0,None,None,1680,17475,6967,55,...,1680,1680,True,True,1251.74,2026-05-28T07:17:38+00:00,2026-05-28T07:38:30+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...


In [30]:
from pathlib import Path
import json
import pandas as pd
from src import history

DATA_ROOT = Path("/home/mdarim/ANTARES_Analysis_Data")
KEEP_DATE = "2026-02-25"

manifest_path = DATA_ROOT / "data" / "lsst_only" / "nightly" / "2026" / "02" / "25" / "manifest.json"
loci_path = DATA_ROOT / "data" / "lsst_only" / "nightly" / "2026" / "02" / "25" / "loci.parquet"
alerts_path = DATA_ROOT / "data" / "lsst_only" / "nightly" / "2026" / "02" / "25" / "alerts.parquet"

with open(manifest_path, "r") as f:
    manifest = json.load(f)

print("Manifest:")
for key in ["date_utc", "mjd_min", "mjd_max", "target_loci", "actual_loci", "alert_rows", "status"]:
    print(key, "=", manifest.get(key))

print("\nFiles exist:")
print("loci:", loci_path.exists(), loci_path)
print("alerts:", alerts_path.exists(), alerts_path)

print("\nParquet row counts:")
print("loci rows:", len(pd.read_parquet(loci_path)))
print("alert rows:", len(pd.read_parquet(alerts_path)))

Manifest:
date_utc = 2026-02-25
mjd_min = 61096.0
mjd_max = 61097.0
target_loci = None
actual_loci = 113459
alert_rows = 1012223
status = complete

Files exist:
loci: True /home/mdarim/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/02/25/loci.parquet
alerts: True /home/mdarim/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/02/25/alerts.parquet

Parquet row counts:
loci rows: 113459
alert rows: 1012223


In [32]:
from pathlib import Path
import shutil
from datetime import datetime, timezone

DATA_ROOT = Path("/home/mdarim/ANTARES_Analysis_Data")
KEEP_PARTS = ("2026", "02", "25")
KEEP_DATE = "2026-02-25"

DRY_RUN = False

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
quarantine = DATA_ROOT / "_quarantine_before_delete" / timestamp

paths_to_move = []

nightly_root = DATA_ROOT / "data" / "lsst_only" / "nightly"
for day_dir in sorted(nightly_root.glob("*/*/*")):
    if day_dir.is_dir() and day_dir.parts[-3:] != KEEP_PARTS:
        paths_to_move.append(day_dir)

cumulative_root = DATA_ROOT / "data" / "lsst_only" / "cumulative"
if cumulative_root.exists():
    paths_to_move.append(cumulative_root)

cache_root = DATA_ROOT / "cache"
if cache_root.exists():
    for item in sorted(cache_root.iterdir()):
        if item.name != "probe50_v1":
            paths_to_move.append(item)
        else:
            for sub in sorted(item.glob("*/*")):
                if sub.is_dir() and sub.name != KEEP_DATE:
                    paths_to_move.append(sub)

print("Will quarantine:")
for p in paths_to_move:
    print(" ", p)

print("\nDestination:")
print(quarantine)

if DRY_RUN:
    print("\nDRY_RUN=True, nothing moved.")
else:
    quarantine.mkdir(parents=True, exist_ok=True)
    for src in paths_to_move:
        if src.exists():
            dst = quarantine / src.relative_to(DATA_ROOT)
            dst.parent.mkdir(parents=True, exist_ok=True)
            print(f"Moving {src} -> {dst}")
            shutil.move(str(src), str(dst))

Will quarantine:
  /home/mdarim/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/02/24
  /home/mdarim/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/02/26
  /home/mdarim/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/02/27
  /home/mdarim/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/02/28
  /home/mdarim/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/05/21
  /home/mdarim/ANTARES_Analysis_Data/data/lsst_only/cumulative
  /home/mdarim/ANTARES_Analysis_Data/cache/chunks
  /home/mdarim/ANTARES_Analysis_Data/cache/full_loci_chunks
  /home/mdarim/ANTARES_Analysis_Data/cache/tile_loci

Destination:
/home/mdarim/ANTARES_Analysis_Data/_quarantine_before_delete/20260525T091133Z
Moving /home/mdarim/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/02/24 -> /home/mdarim/ANTARES_Analysis_Data/_quarantine_before_delete/20260525T091133Z/data/lsst_only/nightly/2026/02/24
Moving /home/mdarim/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/02/26 -> /home/mdarim/ANTARES_Analysis_Data/_quar

In [33]:
from pathlib import Path
from src import history

DATA_ROOT = Path("/home/mdarim/ANTARES_Analysis_Data")

loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

print("Cumulative loci rows:", len(loci_index))
display(nightly_summary)

Cumulative loci rows: 113459


,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,lsst_ss_count,ztf_object_id_count,lsst_only_pass,history_start_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
0,2026-02-25,2026/2/25,61096.0,61097.0,None,None,113459,1012223,12835,5923,...,113459,113459,True,True,27835.51,2026-05-25T00:21:24+00:00,2026-05-25T08:05:19+00:00,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...,/home/mdarim/ANTARES_Analysis_Data/data/lsst_o...
